## Bibliotecas

In [19]:
# Bibliotecas para manipulación de datos
import pandas as pd
import numpy as np
# Bibliotecas para expresiones regulares
import re 
# Silenciar advertencias
import warnings
warnings.filterwarnings("ignore")
# Bibliotecas para estadística y análisis de datos
from sklearn.cluster import KMeans
# Bibliotecas para conexión con Supabase
import ssl
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sqlalchemy.pool import NullPool

## DATASET (Conexión a Supabase)

In [20]:
# Parámetros de conexión a Supabase (Pooler en Transaction Mode: puerto 6543)
import os
from dotenv import load_dotenv
load_dotenv()  # lee el .env del repo (nunca se sube a git)
DB_USER = os.environ["DB_USER"]
DB_PASS = os.environ["DB_PASSWORD"]
DB_HOST = os.environ["DB_HOST"]
DB_PORT = 6543  # pooler en Transaction Mode
DB_NAME = os.environ.get("DB_NAME", "postgres")

# Construcción limpia de la URL de conexión a la base de datos
db_url = URL.create(
    drivername="postgresql+pg8000",
    username=DB_USER,
    password=DB_PASS,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)

# Configuración SSL para el pooler
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

engine = create_engine(
    db_url,
    connect_args={"ssl_context": ssl_ctx},
    poolclass=NullPool
)

print("Conectando a Supabase...")

# Dataset de entrenamiento
query_train = """
    SELECT *
    FROM public.matriz_caracteristicas
    WHERE conjunto = 'ENTRENAMIENTO'
    ORDER BY id_poligono;
"""
df_train = pd.read_sql(query_train, engine)
print(f"Set de Entrenamiento cargado: {df_train.shape} (filas, columnas)")

# Dataset de predicción
query_pred = """
    SELECT *
    FROM public.matriz_caracteristicas
    WHERE conjunto = 'PREDICCION'
    ORDER BY id_poligono;
"""
df_pred = pd.read_sql(query_pred, engine)
print(f"Set de Predicción cargado: {df_pred.shape} (filas, columnas)")

# DataFrame concatenado de entrenamiento y predicción
# Nota: el conjunto de predicción trae el objetivo vacío; se silencia el aviso de pandas al concatenar
with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    df = pd.concat([df_train, df_pred], ignore_index=True)

# Ordenar por id_poligono
df = df.sort_values("id_poligono", ignore_index=True)
print(f"Dataset completo: {df.shape} (filas, columnas)")


Conectando a Supabase...
Set de Entrenamiento cargado: (138, 98) (filas, columnas)
Set de Predicción cargado: (59, 98) (filas, columnas)
Dataset completo: (197, 98) (filas, columnas)


# **Preprocesamiento de Datos**
## 1. Dataset_Modelo
### Selección de columnas (Línea base con ensayo de Random Forest y Ensamble SVR)
Derivado del análisis exploratorio, determinamos las siguientes acciones para un modelo en Random Forest con 50 variables predictoras.

1. Como las 59 parcelas de predicción están en los mismos municipios que el entrenamiento, la CV aleatoria es válida para este split. 
2. Se crea la una nueva variable `grupo_rend` como una variable estratificadora de 5 splits para separar el efecto de zona.
3. No se consideran las variables predictoras con redundancias exactas (`pl_eos20_doy`, `ac_dias_lluvia_ciclo`, `pl_sos20_doy`, `era5_rs_ciclo_mj_m2`, `pend_p90_grados`, `s2_ndvi_int_abr_oct`, `pl_ndvi_pico`, `s2_centro_pico_doy`, `ac_lluvia_llen_mm`, `era5_apar_ciclo_mj_m2`, `ac_lluvia_ciclo_anom_mm`)

Columnas definidas:
- **Metadatos:** `id_poligono`, `municipio`, `estado`, `conjunto`, `clima_pixel_id`
- **Variable objetivo:** `rendimiento_t_ha` (sin transformación: Random Forest no requiere normalidad)
- **Variables predictoras:** 50 variables (se excluyen las redundancias exactas)
- **Variable estratificadora:** `grupo_rend` (KMeans con 2 clusters sobre el rendimiento del set de entrenamiento)


In [21]:
# Variable objetivo
TARGET = "rendimiento_t_ha"

# Variables de metadatos (5 identificadores)
METADATOS = ["id_poligono", "municipio", "estado", "conjunto", "clima_pixel_id"]

# Variables predictoras (51 features)
PREDICTORAS = [
    'area_ha_geod', 'elev_media_m', 'elev_std_m', 'pend_media_grados', 'pend_max_grados', 
    'dz_pixel_m', 'pl_n_picos', 'pl_sos50_doy', 'pl_centro_pico_doy', 'pl_eos50_doy', 
    'pl_dias_meseta', 'pl_caida_intra_ciclo', 'pl_ndvi_base', 'pl_amplitud', 'pl_integral_ciclo', 
    'pl_tasa_subida', 'pl_tasa_bajada', 'pl_ndvi_int_abr_oct', 'pl_lai_max_ciclo', 'pl_lai_int_ciclo', 
    's2_n_picos', 's2_sos50_doy', 's2_ndvi_pico', 's2_integral_ciclo', 'ac_lluvia_pre_mm',
    'ac_lluvia_veg_mm', 'ac_lluvia_ciclo_mm', 'ac_lluvia_max_dia_ciclo_mm', 'ac_racha_seca_max_flor', 'ac_racha_seca_max_ciclo', 
    'ac_gdd_ciclo', 'ac_gdd_hasta_flor', 'ac_balance_ciclo_mm', 'ac_balance_flor_mm', 'ac_balance_llen_mm', 
    'ac_lluvia_ciclo_anom_rel', 'soil_arcilla_pct', 'soil_arena_pct', 'soil_soc_g_kg', 'soil_densidad_g_cm3', 
    'soil_ph', 'soil_limo_pct', 'soil_agua_disp_mm', 'era5_tmax_media_flor_c', 'era5_vpd_medio_flor_kpa', 
    'era5_vpd_medio_llen_kpa', 'era5_sm1_siembra', 'era5_sm2_flor', 'era5_sm2_llen', 'era5_sm2_min_llen', 
    'era5_apar_post_pico_mj_m2'
 ]

# Variable estratificadora (se crea en la siguiente celda)
ESTRATIFICADORA = ["grupo_rend"]

# Redundancias exactas que no entran como predictoras
REDUNDANTES = [
    "pl_eos20_doy", "ac_dias_lluvia_ciclo", "pl_sos20_doy", "era5_rs_ciclo_mj_m2", "pend_p90_grados",
    "s2_ndvi_int_abr_oct", "pl_ndvi_pico", "s2_centro_pico_doy", "ac_lluvia_llen_mm",
    "era5_apar_ciclo_mj_m2", "ac_lluvia_ciclo_anom_mm",
]

COLUMNAS_RF = METADATOS + [TARGET] + PREDICTORAS + ESTRATIFICADORA

# Validaciones de la selección
assert len(PREDICTORAS) == len(set(PREDICTORAS)) == 51, "Predictoras duplicadas o incompletas"
assert not set(PREDICTORAS) & set(REDUNDANTES), "Hay redundancias exactas en las predictoras"
faltantes = [c for c in COLUMNAS_RF if c not in df.columns and c not in ESTRATIFICADORA]
assert not faltantes, f"Columnas no encontradas en el dataset maestro: {faltantes}"
print(f"Columnas seleccionadas: {len(COLUMNAS_RF)} | Predictoras: {len(PREDICTORAS)}")

Columnas seleccionadas: 58 | Predictoras: 51


In [22]:
# Variable estratificadora: clustering del objetivo (KMeans, 2 grupos) sobre el set de entrenamiento
mask_train = df["conjunto"] == "ENTRENAMIENTO"
y_train = df.loc[mask_train, TARGET]
km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(y_train.to_frame())

# El set de predicción no tiene rendimiento, por lo que su grupo queda vacío (NaN)
df["grupo_rend"] = np.nan
df.loc[mask_train, "grupo_rend"] = np.where(km.labels_ == np.argmax(km.cluster_centers_.ravel()), "Alto", "Bajo")
print(df.loc[mask_train].groupby("grupo_rend")[TARGET].describe().round(2))
print("\nUmbral aproximado entre grupos:", round(np.mean(sorted(km.cluster_centers_.ravel())), 2), "t/ha")

            count  mean   std   min   25%   50%   75%   max
grupo_rend                                                 
Alto         80.0  4.60  0.35  3.91  4.45  4.54  4.69  5.59
Bajo         58.0  3.07  0.43  2.00  3.00  3.00  3.50  3.81

Umbral aproximado entre grupos: 3.83 t/ha


In [23]:
# Filtrar el dataset maestro con las columnas de la línea base de Random Forest
df_rf = df[COLUMNAS_RF].copy()

## Verificación rápida

In [24]:
# Verificación rápida del DataFrame resultante
df_rf.info()
print(f"\nDimensiones: {df_rf.shape} (filas, columnas)")
nulos = df_rf.isna().sum()
print(f"\nConteo de nulos por columna:\n{nulos[nulos > 0]}")
print(f"\nTotal de nulos: {nulos.sum()}")
print(f"\nNulos por conjunto:\n{df_rf.drop(columns='conjunto').isna().groupby(df_rf['conjunto']).sum().sum(axis=1)}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197 entries, 0 to 196
Data columns (total 58 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   id_poligono                 197 non-null    object 
 1   municipio                   197 non-null    object 
 2   estado                      197 non-null    object 
 3   conjunto                    197 non-null    object 
 4   clima_pixel_id              197 non-null    object 
 5   rendimiento_t_ha            138 non-null    float64
 6   area_ha_geod                197 non-null    float64
 7   elev_media_m                197 non-null    float64
 8   elev_std_m                  197 non-null    float64
 9   pend_media_grados           197 non-null    float64
 10  pend_max_grados             197 non-null    float64
 11  dz_pixel_m                  197 non-null    float64
 12  pl_n_picos                  197 non-null    int64  
 13  pl_sos50_doy                197 non

## Exportación

In [25]:
# Exportar el dataset limpio a CSV
df_rf.to_csv("dataset_modelo.csv", index=False)
print(f"Archivo exportado: dataset_modelo.csv {df_rf.shape}")

Archivo exportado: dataset_modelo.csv (197, 58)
